# 📈 BCA Stock Predictor — Prediksi Harga Saham BBCA dengan Machine Learning (Python + Streamlit)

Proyek portofolio ini bertujuan untuk memprediksi harga saham **PT Bank Central Asia Tbk (BBCA)** menggunakan data historis dari **Yahoo Finance**, dengan pendekatan Machine Learning.

**Tech Stack:**
- Python 3.10+
- Jupyter Notebook (dijalankan di VS Code)
- `yfinance` — mengambil data harga saham dari Yahoo Finance
- `pandas`, `numpy` — pengolahan data
- `matplotlib`, `seaborn` — visualisasi
- `scikit-learn` — model Machine Learning (Linear Regression, Random Forest + tuning)

> 💡 **Catatan versi:** Notebook ini sengaja tidak memakai TensorFlow/LSTM agar kompatibel dengan Python versi terbaru (termasuk Python 3.13/3.14), karena TensorFlow belum selalu langsung mendukung rilis Python paling baru. Model utamanya adalah **Random Forest** yang sudah di-*tuning*, dengan Linear Regression sebagai baseline pembanding. Kalau kamu memakai Python 3.9–3.12 dan tetap ingin mencoba LSTM, lihat bagian "Rencana Pengembangan" di bawah.

**Alur kerja notebook ini:**
1. Ambil data historis saham BBCA dari Yahoo Finance
2. Eksplorasi data (EDA)
3. Feature engineering (moving average, RSI, lag features, dll.)
4. Split data latih & uji (berbasis waktu / time series split)
5. Melatih beberapa model prediksi (Linear Regression & Random Forest)
6. Tuning hyperparameter Random Forest
7. Evaluasi & perbandingan model
8. Visualisasi hasil prediksi vs harga aktual
9. Prediksi harga beberapa hari ke depan (rekursif)

> ⚠️ **Disclaimer:** Proyek ini dibuat untuk tujuan edukasi & portofolio. Hasil prediksi model **bukan** rekomendasi investasi/trading nyata. Pasar saham dipengaruhi banyak faktor yang tidak sepenuhnya bisa ditangkap oleh data historis.


## 1. Import Library

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use("seaborn-v0_8-darkgrid")
sns.set_palette("viridis")
%matplotlib inline


## 2. Mengambil Data dari Yahoo Finance

Kode saham Bank BCA di Bursa Efek Indonesia (BEI) pada Yahoo Finance adalah **`BBCA.JK`** (akhiran `.JK` menandakan Jakarta Stock Exchange).


In [ ]:
TICKER = "BBCA.JK"
START_DATE = "2015-01-01"
END_DATE = None  # None = sampai hari ini

df = yf.download(TICKER, start=START_DATE, end=END_DATE, auto_adjust=True)

# Versi yfinance terbaru mengembalikan kolom bertingkat (MultiIndex),
# misalnya ('Close', 'BBCA.JK'). Ratakan supaya jadi kolom biasa: 'Close', 'Open', dst.
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df = df.reset_index()

print(f"Jumlah baris data: {len(df)}")
df.head()


In [ ]:
# Simpan data mentah ke folder data/ (opsional, untuk cadangan)
df.to_csv("../data/bbca_raw.csv", index=False)
df.tail()


## 3. Eksplorasi Data (EDA)

In [ ]:
df.info()


In [ ]:
df.describe()


In [ ]:
# Cek missing value
df.isnull().sum()


In [ ]:
plt.figure(figsize=(14,6))
plt.plot(df["Date"], df["Close"], label="Harga Close", color="#0d47a1")
plt.title(f"Harga Penutupan Saham {TICKER} ({df['Date'].min().date()} - {df['Date'].max().date()})")
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.legend()
plt.tight_layout()
plt.savefig("../images/harga_penutupan.png", dpi=150)
plt.show()


In [ ]:
plt.figure(figsize=(14,6))
plt.plot(df["Date"], df["Volume"], color="#6a1b9a")
plt.title(f"Volume Perdagangan Saham {TICKER}")
plt.xlabel("Tanggal")
plt.ylabel("Volume")
plt.tight_layout()
plt.show()


## 4. Feature Engineering

Menambahkan indikator teknikal yang umum dipakai dalam analisis saham:
- **MA7 / MA30 / MA90**: rata-rata bergerak (moving average)
- **Volatility**: standar deviasi return harian
- **RSI (Relative Strength Index)**: indikator momentum
- **Lag features**: harga beberapa hari sebelumnya sebagai fitur prediksi


In [ ]:
data = df[["Date", "Close"]].copy()

# Moving averages
data["MA7"] = data["Close"].rolling(window=7).mean()
data["MA30"] = data["Close"].rolling(window=30).mean()
data["MA90"] = data["Close"].rolling(window=90).mean()

# Return harian & volatilitas
data["Return"] = data["Close"].pct_change()
data["Volatility30"] = data["Return"].rolling(window=30).std()

# RSI (14 hari)
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(window=period).mean()
    avg_loss = loss.rolling(window=period).mean()
    rs = avg_gain / (avg_loss + 1e-9)
    return 100 - (100 / (1 + rs))

data["RSI14"] = compute_rsi(data["Close"])

# Lag features (harga 1, 2, 3 hari sebelumnya)
for lag in [1, 2, 3, 5]:
    data[f"Close_lag{lag}"] = data["Close"].shift(lag)

data = data.dropna().reset_index(drop=True)
data.head()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14,9), sharex=True)

axes[0].plot(data["Date"], data["Close"], label="Close", color="#0d47a1")
axes[0].plot(data["Date"], data["MA7"], label="MA7", alpha=0.8)
axes[0].plot(data["Date"], data["MA30"], label="MA30", alpha=0.8)
axes[0].plot(data["Date"], data["MA90"], label="MA90", alpha=0.8)
axes[0].set_title(f"Harga & Moving Average {TICKER}")
axes[0].legend()

axes[1].plot(data["Date"], data["RSI14"], color="#e65100")
axes[1].axhline(70, color="red", linestyle="--", alpha=0.6)
axes[1].axhline(30, color="green", linestyle="--", alpha=0.6)
axes[1].set_title("RSI (14 Hari)")

plt.tight_layout()
plt.savefig("../images/technical_indicators.png", dpi=150)
plt.show()


## 5. Split Data Latih & Uji

Untuk data time series, split dilakukan **berdasarkan urutan waktu** (bukan acak/random), agar model tidak "mengintip" masa depan.


In [ ]:
feature_cols = ["MA7", "MA30", "MA90", "Volatility30", "RSI14",
                "Close_lag1", "Close_lag2", "Close_lag3", "Close_lag5"]
target_col = "Close"

X = data[feature_cols].values
y = data[target_col].values
dates = data["Date"].values

split_idx = int(len(data) * 0.85)

X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
dates_train, dates_test = dates[:split_idx], dates[split_idx:]

print(f"Data latih : {len(X_train)} baris")
print(f"Data uji   : {len(X_test)} baris")


## 6. Model Machine Learning (Baseline)

Dua model klasik sebagai pembanding sebelum masuk ke LSTM.


In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    results[name] = {"RMSE": rmse, "MAE": mae, "MAPE (%)": mape, "R2": r2}
    print(f"[{name}] RMSE={rmse:.2f} | MAE={mae:.2f} | MAPE={mape:.2f}% | R2={r2:.4f}")


In [ ]:
# --- Linear Regression ---
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
evaluate("Linear Regression", y_test, lr_pred)


In [ ]:
# --- Random Forest ---
rf_model = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
evaluate("Random Forest", y_test, rf_pred)


## 7. Tuning Random Forest (Model Utama)

Random Forest dipilih sebagai model utama karena robust terhadap noise, tidak butuh scaling, dan performanya kompetitif untuk data tabular seperti fitur teknikal saham. Di sini dilakukan pencarian kombinasi hyperparameter terbaik dengan `GridSearchCV` (validasi berbasis time series, bukan k-fold acak, agar tidak bocor informasi masa depan).


In [ ]:
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

param_grid = {
    "n_estimators": [200, 300, 500],
    "max_depth": [5, 8, 12, None],
    "min_samples_leaf": [1, 2, 5],
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_grid=param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Parameter terbaik:", grid_search.best_params_)
best_rf_model = grid_search.best_estimator_


In [ ]:
best_rf_pred = best_rf_model.predict(X_test)
evaluate("Random Forest (Tuned)", y_test, best_rf_pred)


In [ ]:
# Feature importance — fitur mana yang paling berpengaruh terhadap prediksi
importances = pd.Series(best_rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(8,5))
importances.plot(kind="barh", color="#2e7d32")
plt.title("Feature Importance — Random Forest (Tuned)")
plt.xlabel("Tingkat Kepentingan")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("../images/feature_importance.png", dpi=150)
plt.show()


## 8. Perbandingan Semua Model

In [ ]:
results_df = pd.DataFrame(results).T
results_df.sort_values("RMSE")


In [ ]:
results_df["RMSE"].sort_values().plot(kind="barh", figsize=(8,4), color="#00695c")
plt.title("Perbandingan RMSE Antar Model (semakin kecil semakin baik)")
plt.xlabel("RMSE")
plt.tight_layout()
plt.savefig("../images/model_comparison.png", dpi=150)
plt.show()


## 9. Visualisasi Prediksi vs Harga Aktual (Model Terbaik: Random Forest Tuned)

In [ ]:
plt.figure(figsize=(14,6))
plt.plot(dates_test, y_test, label="Harga Aktual", color="#0d47a1")
plt.plot(dates_test, best_rf_pred, label="Prediksi Random Forest (Tuned)", color="#e65100", linestyle="--")
plt.title(f"Prediksi vs Aktual Harga Saham {TICKER} (Data Uji)")
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.legend()
plt.tight_layout()
plt.savefig("../images/prediksi_vs_aktual.png", dpi=150)
plt.show()


## 10. Prediksi Harga N Hari ke Depan

Prediksi dilakukan secara **rekursif** menggunakan Random Forest yang sudah di-tuning: fitur (MA, RSI, lag, dst.) dihitung ulang setiap langkah menggunakan harga hasil prediksi sebelumnya, lalu dipakai untuk memprediksi hari berikutnya.


In [ ]:
N_DAYS_FUTURE = 7

# Jaga-jaga: pastikan tidak ada sisa kolom MultiIndex dari yfinance
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

history_df = data[["Date", "Close"]].copy()
history_df["Date"] = pd.to_datetime(history_df["Date"])
history_df["Close"] = history_df["Close"].astype(float)

future_rows = []

for _ in range(N_DAYS_FUTURE):
    tmp = history_df.copy()
    tmp["MA7"] = tmp["Close"].rolling(window=7).mean()
    tmp["MA30"] = tmp["Close"].rolling(window=30).mean()
    tmp["MA90"] = tmp["Close"].rolling(window=90).mean()
    tmp["Return"] = tmp["Close"].pct_change()
    tmp["Volatility30"] = tmp["Return"].rolling(window=30).std()
    tmp["RSI14"] = compute_rsi(tmp["Close"])
    for lag in [1, 2, 3, 5]:
        tmp[f"Close_lag{lag}"] = tmp["Close"].shift(lag)

    last_features = tmp[feature_cols].iloc[[-1]]
    next_pred_price = float(best_rf_model.predict(last_features)[0])

    next_date = pd.bdate_range(start=history_df["Date"].iloc[-1] + pd.Timedelta(days=1), periods=1)[0]
    future_rows.append({"Tanggal": next_date, "Prediksi_Close": next_pred_price})

    # Tambahkan hasil prediksi ke history_df supaya jadi input hari berikutnya
    new_row = pd.DataFrame({"Date": [next_date], "Close": [next_pred_price]})
    history_df = pd.concat([history_df, new_row], ignore_index=True)

future_df = pd.DataFrame(future_rows)
future_df


In [ ]:
plt.figure(figsize=(14,6))
plt.plot(data["Date"].values[-60:], data["Close"].values[-60:], label="Historis (60 hari terakhir)", color="#0d47a1")
plt.plot(future_df["Tanggal"], future_df["Prediksi_Close"], label=f"Prediksi {N_DAYS_FUTURE} Hari ke Depan", color="#c62828", marker="o", linestyle="--")
plt.title(f"Prediksi Harga {TICKER} — {N_DAYS_FUTURE} Hari ke Depan")
plt.xlabel("Tanggal")
plt.ylabel("Harga (Rp)")
plt.legend()
plt.tight_layout()
plt.savefig("../images/prediksi_masa_depan.png", dpi=150)
plt.show()


## 11. Menyimpan Model (opsional)

In [ ]:
import joblib

joblib.dump(best_rf_model, "../models/random_forest_bbca_model.pkl")
print("Model tersimpan di ../models/random_forest_bbca_model.pkl")



**Catatan:** Notebook ini dibuat sebagai proyek portofolio dan sarana pembelajaran data science / machine learning di Python, bukan alat prediksi finansial yang siap dipakai untuk keputusan investasi nyata.
